In [1]:
# Run this cell first (kernel: project .venv — `uv sync --group dev`).
import importlib.util
import json
import os
import re
import sys
from pathlib import Path

import pandas as pd
import requests
from datasets import (
    get_dataset_config_names,
    get_dataset_split_names,
    load_dataset,
)
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import display


def _locate_load_env_py() -> Path:
    starts: list[Path] = [Path.cwd().resolve()]
    if "__vsc_ipynb_file__" in globals():
        starts.insert(0, Path(__vsc_ipynb_file__).resolve().parent)
    seen: set[Path] = set()
    for start in starts:
        for candidate in [start, *start.parents]:
            if candidate in seen:
                continue
            seen.add(candidate)
            env_file = candidate / "load_env.py"
            if env_file.is_file():
                return env_file
    raise FileNotFoundError(
        "load_env.py not found. cd to the Romcom-Drift repo root "
        "(or clone it) before running this notebook."
    )


_load_env_path = _locate_load_env_py()
_spec = importlib.util.spec_from_file_location("load_env", _load_env_path)
if _spec is None or _spec.loader is None:
    raise ImportError(f"Cannot load {_load_env_path}")
load_env = importlib.util.module_from_spec(_spec)
sys.modules["load_env"] = load_env
_spec.loader.exec_module(load_env)

PROJECT_DIR = load_env.bootstrap_project()
api = HfApi()

print("PROJECT_DIR:", PROJECT_DIR)
print("Python:", sys.executable)
print("HF_TOKEN configured:", load_env.huggingface_token_configured())

for folder in [
    "data/raw",
    "data/processed",
    "data/metadata",
    "outputs",
    "logs",
]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

PROJECT_DIR: /home/shrys/dev/Romcom-Drift
Python: /home/shrys/dev/Romcom-Drift/.venv/bin/python
HF_TOKEN configured: True


In [2]:
# auditing datasets, repo metadata
api = HfApi()

DATASETS = [
    "Divyanshu/indicxnli",
    "ai4bharat/IndicXParaphrase",
    "ai4bharat/indic-align",
]

dataset_inventory = []

for repo in DATASETS:

    print(f"\nInspecting: {repo}")

    try:
        info = api.dataset_info(
            repo_id=repo,
            files_metadata=False,
        )

        card = info.cardData or {}

        files = [
            file.rfilename
            for file in (info.siblings or [])
        ]

        result = {
            "repository": repo,
            "revision": info.sha,
            "license": card.get("license"),
            "number_of_files": len(files),
            "files": files[:15],
            "status": "ACCESSIBLE",
        }

        dataset_inventory.append(result)

        print(json.dumps(result, indent=2))

    except Exception as e:

        print("ACCESS ERROR:", str(e))

        dataset_inventory.append({
            "repository": repo,
            "status": "FAILED",
            "error": str(e),
        })


Inspecting: Divyanshu/indicxnli
{
  "repository": "Divyanshu/indicxnli",
  "revision": "7092c27872e919f31d0496fb8b9c47bd2cba3f6c",
  "license": [
    "cc0-1.0"
  ],
  "number_of_files": 59,
  "files": [
    ".gitattributes",
    ".gitignore",
    "README.md",
    "backward/dev/xnli_as.json",
    "backward/dev/xnli_bn.json",
    "backward/dev/xnli_gu.json",
    "backward/dev/xnli_hi.json",
    "backward/dev/xnli_kn.json",
    "backward/dev/xnli_ml.json",
    "backward/dev/xnli_mr.json",
    "backward/dev/xnli_or.json",
    "backward/dev/xnli_pa.json",
    "backward/dev/xnli_ta.json",
    "backward/dev/xnli_te.json",
    "backward/test/xnli_as.json"
  ],
  "status": "ACCESSIBLE"
}

Inspecting: ai4bharat/IndicXParaphrase
{
  "repository": "ai4bharat/IndicXParaphrase",
  "revision": "181031f9e69c465e0c72fc84fd33b66114adb3e8",
  "license": null,
  "number_of_files": 12,
  "files": [
    ".gitattributes",
    "IndicXParaphrase.py",
    "data/as.tsv",
    "data/bn.tsv",
    "data/gu.tsv",
  

In [3]:
DAKSHINA_URL = (
    "https://storage.googleapis.com/gresearch/dakshina/"
    "dakshina_dataset_v1.0.tar"
)

response = requests.head(
    DAKSHINA_URL,
    allow_redirects=True,
    timeout=30,
)

print("HTTP status:", response.status_code)

size = response.headers.get("Content-Length")

if size:
    print("Archive size:", round(int(size) / 1024**2, 2), "MB")
else:
    print("Archive size: Not reported")

print("Content type:", response.headers.get("Content-Type"))

HTTP status: 200
Archive size: 1915.3 MB
Content type: application/x-tar


In [4]:
audit_path = (
    PROJECT_DIR / "data/metadata/dataset_inventory.json"
)

audit_path.write_text(
    json.dumps(dataset_inventory, indent=2),
    encoding="utf-8",
)

print("Inventory saved:", audit_path)

Inventory saved: /home/shrys/dev/Romcom-Drift/data/metadata/dataset_inventory.json


In [5]:
# Dataset probe cells use imports and `api` from the setup cell above.

In [7]:
repo = "Divyanshu/indicxnli"

info = api.dataset_info(repo)

layout = sorted(
    {
        sibling.rfilename.split("/")[0]
        for sibling in (info.siblings or [])
        if "/" in sibling.rfilename
    }
)
langs = sorted(
    {
        match.group(1)
        for sibling in (info.siblings or [])
        for match in [re.search(r"xnli_(\w+)\.json", sibling.rfilename)]
        if match
    }
)

print("TOP-LEVEL LAYOUT:", layout)
print("LANGUAGE CODES (from filenames):", langs)

print("\nHINDI TRAIN FILES:")
for sibling in info.siblings or []:
    name = sibling.rfilename
    if "hi" in name.lower() and "train" in name.lower():
        print(name)

TOP-LEVEL LAYOUT: ['backward', 'forward']
LANGUAGE CODES (from filenames): ['as', 'bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'or', 'pa', 'ta', 'te']

HINDI TRAIN FILES:
forward/train/xnli_hi.json


In [ ]:
repo = "ai4bharat/IndicXParaphrase"

file_path = hf_hub_download(
    repo_id=repo,
    filename="data/hi.tsv",
    repo_type="dataset"
)

df = pd.read_csv(file_path, sep="\t")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head(5))

print("\nLabel distribution:")
print(df["label"].value_counts(dropna=False))

hi.tsv:   0%|          | 0.00/1.21M [00:00<?, ?B/s]

Shape: (2002, 4)

Columns:
['en', 'sentence1', 'sentence2', 'label']


,en,sentence1,sentence2,label
0,"on receiving the information, the police reach...",सूचना मिलते ही पुलिस मौके पर पहुंची और घायलों ...,पुलिस मौके पर सूचना मिलते ही पहुंची और घायलों ...,1
1,"on receiving the information, the police reach...",सूचना मिलते ही पुलिस मौके पर पहुंची और घायलों ...,सूचना मिलते ही घायलों को स्थानीय अस्पताल में भ...,0
2,he won the filmfare award for best supporting ...,इस फिल्म में उनके अभिनय के लिए उन्हें सर्वश्रे...,सर्वश्रेष्ठ सहायक अभिनेता का फिल्मफेयर पुरस्का...,1
3,he won the filmfare award for best supporting ...,इस फिल्म में उनके अभिनय के लिए उन्हें सर्वश्रे...,उनके अभिनय के लिए उन्हें सर्वश्रेष्ठ सहायक अभि...,0
4,chandigarh: punjab kala parishad would honour ...,चंडीगढ़ः पंजाब कला परिषद प्रख्यात पंजाबी साहित...,चंडीगढ़ः प्रख्यात पंजाबी साहित्यकार जसवंत सिं...,1



Label distribution:
label
1    1001
0    1001
Name: count, dtype: int64


In [ ]:
repo = "ai4bharat/indic-align"

configs = get_dataset_config_names(repo)

print("AVAILABLE CONFIGURATIONS:")
print(configs)

toxic_config = next(
    (c for c in configs if "toxic" in c.lower() and "matrix" in c.lower()),
    None
)

assert toxic_config is not None, "Toxic Matrix config not found"

splits = get_dataset_split_names(
    repo,
    config_name=toxic_config
)

print("\nSplits:", splits)

ds = load_dataset(
    repo,
    name=toxic_config,
    split=splits[0],
    streaming=True
)

print("\nFeatures:", ds.features)

for i, row in enumerate(ds.take(3)):
    print(f"\nEXAMPLE {i+1}")

    for key, value in row.items():
        print(f"{key}: {str(value)[:500]}")

README.md:   0%|          | 0.00/4.98k [00:00<?, ?B/s]

AVAILABLE CONFIGURATIONS:
['Indic_ShareLlama', 'Dolly_T', 'OpenAssistant_T', 'WikiHow', 'IndoWordNet', 'Anudesh', 'Wiki_Conv', 'Wiki_Chat', 'HHRLHF_T', 'Toxic_Matrix']

Splits: ['train']

Features: {'doc_id': Value('string'), 'num_turns': Value('float64'), 'eng_Latn': List(List(Value('string'))), 'asm_Beng': List(List(Value('string'))), 'asm_Latn': List(List(Value('string'))), 'ben_Beng': List(List(Value('string'))), 'ben_Latn': List(List(Value('string'))), 'guj_Gujr': List(List(Value('string'))), 'hin_Deva': List(List(Value('string'))), 'hin_Latn': List(List(Value('string'))), 'kan_Knda': List(List(Value('string'))), 'kan_Latn': List(List(Value('string'))), 'mal_Mlym': List(List(Value('string'))), 'mal_Latn': List(List(Value('string'))), 'mar_Deva': List(List(Value('string'))), 'mar_Latn': List(List(Value('string'))), 'npi_Deva': List(List(Value('string'))), 'npi_Latn': List(List(Value('string'))), 'ory_Orya': List(List(Value('string'))), 'ory_Latn': List(List(Value('string'))), 'pan_

In [ ]:
repo = "LingoIITGN/COMI-LINGUA"

configs = get_dataset_config_names(repo)

print("CONFIGURATIONS:")
print(configs)

for config_name in configs:

    print(f"\n{'='*50}")
    print("CONFIG:", config_name)

    splits = get_dataset_split_names(
        repo,
        config_name=config_name
    )

    print("Splits:", splits)

    ds = load_dataset(
        repo,
        name=config_name,
        split="train",
        streaming=True
    )

    print("Features:", ds.features)

    for i, row in enumerate(ds.take(2)):

        print(f"\nSample {i+1}")

        for key, value in row.items():
            print(f"{key}: {str(value)[:400]}")

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

CONFIGURATIONS:
['LID', 'MLI', 'POS', 'NER', 'TN', 'MT']

CONFIG: LID
Splits: ['train', 'test']
Features: None

Sample 1
Sentences: इस स्पोकन ट्यूटोरियल में हम usb driver संस्थापित करने के लिए कंप्यूटर और sbhs के बीच फिज़िकल कम्युनिकेशन सेटअप करना सीखेंगे|
Predicted tags: [{'key': 'इस', 'value': 'hi', 'index': 0}, {'key': 'स्पोकन', 'value': 'hi', 'index': 1}, {'key': 'ट्यूटोरियल', 'value': 'hi', 'index': 2}, {'key': 'में', 'value': 'hi', 'index': 3}, {'key': 'हम', 'value': 'hi', 'index': 4}, {'key': 'usb', 'value': 'en', 'index': 5}, {'key': 'driver', 'value': 'en', 'index': 6}, {'key': 'संस्थापित', 'value': 'hi', 'index': 7}, {'key': 'करने', 'value': 'hi', 'index'
Annotated by: Annotator 1: [{'key': 'इस', 'value': 'hi', 'index': 0}, {'key': 'स्पोकन', 'value': 'en', 'index': 1}, {'key': 'ट्यूटोरियल', 'value': 'en', 'index': 2}, {'key': 'में', 'value': 'hi', 'index': 3}, {'key': 'हम', 'value': 'hi', 'index': 4}, {'key': 'usb', 'value': 'en', 'index': 5}, {'key': 'driver', 'value': 'en',

In [ ]:
from datasets import load_dataset
import pandas as pd

repo = "LingoIITGN/COMI-LINGUA"

mli = load_dataset(
    repo,
    name="MLI",
    split="train"
)

print("Rows:", len(mli))
print("\nColumns:")
print(mli.column_names)

print("\nFeatures:")
print(mli.features)

for i in range(5):
    row = mli[i]

    print(f"\nSAMPLE {i+1}")

    for key, value in row.items():
        if isinstance(value, str):
            print(f"{key}: {value[:250]}")
        else:
            print(f"{key}: {str(value)[:250]}")

In [ ]:
roman_fields = [
    col for col in mli.column_names
    if "rh" in col.lower()
]

print("Romanized Hindi fields:", roman_fields)

for field in roman_fields:

    values = [
        x for x in mli[field]
        if isinstance(x, str) and x.strip()
    ]

    print(
        field,
        "| Non-empty:", len(values),
        "| Unique:", len(set(values))
    )

    print("Examples:")
    for x in values[:5]:
        print("-", x)

In [ ]:
tn = load_dataset(
    repo,
    name="TN",
    split="train"
)

print("Rows:", len(tn))
print("Columns:", tn.column_names)
print("Features:", tn.features)

for i in range(5):
    print("\n", tn[i])